# 05 - Train Damage Classifier with DINOv2

Kaggle notebook for training building-level damage classification using DINOv2 backbone.
Two-stage training: frozen backbone (stage 1) then partial unfreezing (stage 2).
Uses pre/post building crops with FocalLoss for class imbalance.

In [ ]:
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    # Attach xBD dataset + processed_xbd dataset to this notebook
    PROCESSED_DIR = Path('/kaggle/input/processed-xbd/processed_xbd')  # Adjust dataset slug
    OUTPUT_DIR = Path('/kaggle/working')
    sys.path.insert(0, '/kaggle/working')
    !pip install -q transformers albumentations
else:
    PROCESSED_DIR = Path('../data/processed/xbd')
    OUTPUT_DIR = Path('../outputs')
    sys.path.insert(0, str(Path('../src').resolve()))

print(f'Processed dir: {PROCESSED_DIR} (exists: {PROCESSED_DIR.exists()})')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
import torch
from torch.utils.data import DataLoader
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

from rescuevision.data.datasets import XBDDamageCropDataset
from rescuevision.data.transforms import get_classification_transforms
from rescuevision.models.dinov2_classifier import DINOv2Classifier
from rescuevision.models.losses import FocalLoss, compute_class_weights
from rescuevision.training.train_damage_classifier import train_damage_classifier
from rescuevision.utils.seed import set_seed

## Configuration

In [ ]:
config = {
    "seed": 42,
    "device": "cuda",
    "dataset": {
        "processed_dir": str(PROCESSED_DIR),
        "split_csv": str(PROCESSED_DIR / "metadata" / "splits_event_aware.csv"),
        "crops_csv": str(PROCESSED_DIR / "metadata" / "building_crops.csv"),
        "image_size": 224,
        "batch_size": 32,
        "num_workers": 2,
    },
    "model": {
        "backbone": "facebook/dinov2-base",
        "num_classes": 4,
        "freeze_backbone": True,
        "mlp_hidden_dim": 512,
        "dropout": 0.3,
        "unfreeze_after_epoch": 10,
        "unfreeze_blocks": 4,
    },
    "training": {
        "epochs": 30,
        "lr": 1e-3,
        "stage2_lr": 1e-5,
        "weight_decay": 1e-4,
        "scheduler": "cosine",
        "mixed_precision": True,
        "early_stopping_patience": 7,
        "save_best_metric": "macro_f1",
    },
    "loss": {
        "focal_gamma": 2.0,
    },
    "augmentation": {
        "horizontal_flip": 0.5,
        "vertical_flip": 0.5,
        "rotate": 15,
        "brightness_contrast": 0.2,
        "normalize": True,
    },
}

set_seed(config["seed"])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')

## Create Datasets

In [ ]:
train_transform = get_classification_transforms(
    config["augmentation"], phase="train", image_size=224,
)
val_transform = get_classification_transforms(
    config["augmentation"], phase="val", image_size=224,
)

train_dataset = XBDDamageCropDataset(
    crops_csv=config["dataset"]["crops_csv"],
    split_csv=config["dataset"]["split_csv"],
    split="train",
    transform=train_transform,
    image_size=224,
)

val_dataset = XBDDamageCropDataset(
    crops_csv=config["dataset"]["crops_csv"],
    split_csv=config["dataset"]["split_csv"],
    split="val",
    transform=val_transform,
    image_size=224,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config["dataset"]["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config["dataset"]["batch_size"],
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f'Train: {len(train_dataset)} crops, Val: {len(val_dataset)} crops')

## Compute Class Weights

In [ ]:
import numpy as np

# Collect labels from the training dataset for class weight computation
train_labels = []
for i in range(len(train_dataset)):
    sample = train_dataset[i]
    train_labels.append(sample["label"])

class_weights = compute_class_weights(train_labels, num_classes=4)
class_weights = class_weights.to(device)

label_names = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
for name, weight in zip(label_names, class_weights):
    print(f'  {name}: weight={weight:.4f}')

## Create Model and Train

In [ ]:
model = DINOv2Classifier(
    backbone=config["model"]["backbone"],
    num_classes=config["model"]["num_classes"],
    freeze_backbone=config["model"]["freeze_backbone"],
    mlp_hidden_dim=config["model"]["mlp_hidden_dim"],
    dropout=config["model"]["dropout"],
)

loss_fn = FocalLoss(
    gamma=config["loss"]["focal_gamma"],
    alpha=class_weights,
)

output_dir = OUTPUT_DIR / "damage_classifier_dinov2"

best_metrics = train_damage_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    config=config,
    output_dir=str(output_dir),
    class_names=label_names,
)

print(f'\nBest metrics: {best_metrics}')

## Visualize Predictions

In [ ]:
import matplotlib.pyplot as plt

model.eval()
model.to(device)

fig, axes = plt.subplots(2, 6, figsize=(24, 8))

# Get a batch of validation samples
batch = next(iter(val_loader))
pre_crops = batch["pre_crop"][:6].to(device)
post_crops = batch["post_crop"][:6].to(device)
labels = batch["label"][:6].numpy()

with torch.no_grad():
    logits = model(pre_crops, post_crops)
    preds = logits.argmax(dim=1).cpu().numpy()

for i in range(6):
    # Denormalize pre-disaster crop
    pre_img = pre_crops[i].cpu().numpy().transpose(1, 2, 0)
    pre_img = (pre_img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    pre_img = np.clip(pre_img, 0, 1)
    
    # Denormalize post-disaster crop
    post_img = post_crops[i].cpu().numpy().transpose(1, 2, 0)
    post_img = (post_img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    post_img = np.clip(post_img, 0, 1)
    
    axes[0, i].imshow(pre_img)
    axes[0, i].set_title(f'Pre | True: {label_names[labels[i]]}', fontsize=9)
    axes[1, i].imshow(post_img)
    axes[1, i].set_title(f'Post | Pred: {label_names[preds[i]]}', fontsize=9)
    
    # Color border: green=correct, red=wrong
    color = 'green' if preds[i] == labels[i] else 'red'
    for spine in axes[1, i].spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)
    
    for ax in axes[:, i]:
        ax.axis('off')

plt.suptitle('Damage Classification: Pre/Post Crops with Predictions', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Confusion Matrix

In [ ]:
from rescuevision.evaluation.metrics_classification import plot_classification_confusion_matrix

# The confusion matrix is already saved by train_damage_classifier,
# but we can display it here as well.
import seaborn as sns
from sklearn.metrics import confusion_matrix

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in val_loader:
        pre = batch["pre_crop"].to(device)
        post = batch["post_crop"].to(device)
        logits = model(pre, post)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch["label"].numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

plot_classification_confusion_matrix(
    all_labels,
    all_preds,
    class_names=label_names,
    title='DINOv2 Damage Classification Confusion Matrix',
    save_path=output_dir / 'confusion_matrix_display.png',
)
print('Confusion matrix saved.')

## Classification Report

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    all_labels, all_preds,
    target_names=label_names,
    zero_division=0,
)
print(report)

## Save Checkpoint

In [ ]:
if IS_KAGGLE:
    print('Checkpoint saved at:', output_dir / 'checkpoints')
    print('Save this output as a Kaggle dataset for the inference notebook.')
    !ls -la {output_dir}/checkpoints/
else:
    print('Training complete. Checkpoint saved.')